In [1]:
import pandas as pd
def read_cts_file(filepath):
    """Reads a .cts file, skipping header lines starting with # or CTS."""
    skip_rows = 0
    with open(filepath, 'r') as f:
        for line in f:
            if line.startswith('#') or line.startswith('CTS'):
                skip_rows += 1
            else:
                break
    return pd.read_csv(filepath, sep='\t', skiprows=skip_rows,
                       decimal=',', na_values=['NaN'])
def trim_by_derivative(toco_signal, threshold=1.0):
    """
    Finds the start of real data by detecting the first significant
    change in the TOCO signal.
    """
    diffs = np.abs(np.diff(toco_signal, prepend=toco_signal[0]))
    change_points = np.where(diffs > threshold)[0]
    return change_points[0] + 240 if len(change_points) > 0 else 0

In [2]:
import numpy as np
from scipy.signal import butter, filtfilt, find_peaks


# ── Filtering ─────────────────────────────────────────────────────────────────

def apply_butterworth_filter(data, cutoff=0.05, fs=4.0, order=2):
    """
    Applies a low-pass Butterworth filter to smooth the TOCO signal.
    cutoff: frequency above which signals are attenuated (Hz)
    fs:     sampling frequency (Hz) — 4 Hz means 1 sample per 0.25s
    order:  steepness of the filter rolloff
    """
    nyquist     = 0.5 * fs
    normal_cut  = cutoff / nyquist
    b, a        = butter(order, normal_cut, btype='low', analog=False)
    return filtfilt(b, a, data)


# ── Segmentation ──────────────────────────────────────────────────────────────

def get_toco_segments(raw_toco, toco_filtered, fs=4.0):
    """
    Splits the TOCO signal at calibration jumps (sudden ≥10 mmHg steps).
    Each segment gets its own baseline (10th percentile of filtered signal).
    Segments shorter than 2 minutes are ignored.
    """
    segments       = []
    last_idx       = 0
    min_seg_len    = 120 * fs
    total_len      = len(raw_toco)

    for i in range(1, total_len):
        is_jump        = abs(raw_toco[i] - raw_toco[i - 1]) >= 10
        is_long_enough = (i - last_idx) >= min_seg_len

        if is_jump and is_long_enough:
            segment_data = toco_filtered[last_idx:i]
            if len(segment_data) > 0:
                segments.append(_make_segment(last_idx, i, segment_data, fs))
            last_idx = i

    # Last segment
    final_data = toco_filtered[last_idx:]
    if len(final_data) > 0:
        segments.append(_make_segment(last_idx, total_len - 1, final_data, fs))

    return segments


def _make_segment(start, end, data, fs):
    """Builds a segment dict. Private — used by get_toco_segments."""
    return {
        'indices':      (start, end),
        'time_seconds': (start / fs, end / fs),
        'baseline':     np.nanpercentile(data, 10)
    }


# ── Contraction detection ─────────────────────────────────────────────────────

def find_contractions(toco_filtered, segments, threshold=15, fs=4.0, time_threshold=2):
    """
    Detects uterine contractions within each TOCO segment.
    Steps:
      1. Find local peaks above threshold
      2. Merge close peaks / split at deep valleys (50% rule)
      3. Find start/end boundaries around each peak group
      4. Resolve any overlapping boundaries
    """
    toco_filtered = np.array(toco_filtered, copy=True)
    contractions  = []
    peaks_save = []

    starts_seg = np.array([d['indices'][0] for d in segments])
    ends_seg  = np.array([d['indices'][1] for d in segments])

    peaks = _find_valid_peaks(toco_filtered, starts_seg, ends_seg, segments, threshold)
    groups = _group_peaks(toco_filtered, peaks, starts_seg, ends_seg, segments)

    seg_contractions = []
    for group in groups:
        start_peak = group[0]
        end_peak = group[1]

        result = _find_start_end(
            toco_filtered, starts_seg, ends_seg, segments, time_threshold, start_peak, end_peak, fs
        )
        if result is None:
            continue
        start, end = result

        duration_s = (end - start) / fs

        seg_contractions.append({
            "start_seconds": start / fs,
            "end_seconds":   end / fs,
            "start_idx":     start,
            "end_idx":       end,
            "duration":      duration_s,
            #"peak_amplitude": float(np.max(toco_filtered[start:end]) - baseline),
            "peak_s":        start_peak / fs
        })

    seg_contractions = _resolve_overlaps(seg_contractions, toco_filtered, fs)
    contractions.extend(seg_contractions)
    #print(seg_contractions)

    return contractions


#get all the values that is part of a contraction, not just the values inside the segment
def _find_valid_peaks(toco_filtered, starts, ends, segments, threshold, fs=4.0):
    peaks = []
    peaks_scipy, _ = find_peaks(toco_filtered)

    starts = np.array([d['indices'][0] for d in segments])
    ends   = np.array([d['indices'][1] for d in segments])

    for p in peaks_scipy:
        mask = (starts <= p) & (ends > p)
        matching = [d for d, m in zip(segments, mask) if m]

        if not matching:
            continue

        baseline = matching[0]['baseline']

        if toco_filtered[p] >= baseline + threshold:
            peaks.append(p)
    #print("FIND VALID PEAKS: ", peaks)
    return peaks


def _group_peaks(toco_filtered, peaks, starts_segs, ends_segs, segments, percentage_threshold=0.5, drop_threshold=15, fs=4.0):
    """
    Groups adjacent peaks that belong to the same contraction.
    Splits into a new group if the valley between them drops below a certain threshold
    """
    if len(peaks) == 0:
        return []

    def get_baseline(idx):
        mask = (starts_segs <= idx) & (ends_segs > idx)
        matching_indices = np.where(mask)[0]
        if len(matching_indices) > 0:
            return segments[matching_indices[0]]['baseline']
        return np.nanpercentile(toco_filtered, 10)

    groups = []
    i = 0

    while i < len(peaks):
        start_peak = peaks[i]
        end_peak = peaks[i]

        j = i + 1
        while j < len(peaks):
            next_peak = peaks[j]

            # Find the deepest valley between the current end_peak and the next_peak
            valley_min = np.min(toco_filtered[end_peak : next_peak + 1])
            baseline = get_baseline(end_peak)

            peak_amp = toco_filtered[end_peak] - baseline

            drop_amount = toco_filtered[end_peak] - valley_min

            if toco_filtered[next_peak] - valley_min >= drop_threshold or toco_filtered[peaks[j-1]] - valley_min >= drop_threshold:
                break
            else:
                end_peak = next_peak
                j += 1

        groups.append([start_peak, end_peak])
        i = j
    #print("GROUPS: ", groups)
    #print()
    return groups

def _find_start_end(signal, starts, ends, segments, time_threshold, start_peak, end_peak, fs=4.0):
    """
    Walks left and right from peak_idx to find where the contraction
    begins and ends (signal returning close to baseline).
    Private — used by find_contractions.
    """
    # Search LEFT
    start_idx = start_peak
    mask = (starts <= start_peak) & (ends > start_peak)

    matching = [d for d, m in zip(segments, mask) if m]
#    print(matching)

    if not matching:
        return

    baseline = matching[0]['baseline']
    seg_start = max(0, int(matching[0]['indices'][0] - 60 * fs))
    seg_end   = min(len(signal) - 1, int(matching[0]['indices'][1] + 60 * fs))

    peak_amp = signal[start_peak] - baseline

    while start_idx > seg_start:
        if signal[start_idx] <= baseline + 5:
            break
        start_idx -= 1

    # Search RIGHT
    end_idx = end_peak

    while end_idx < seg_end - 1:
        if signal[end_idx] <= baseline + 5:
            break
        end_idx += 1
    if start_idx != start_peak and end_idx != end_peak:
        start_idx = start_idx + np.argmin(signal[start_idx : start_peak])
        end_idx = end_peak + np.argmin(signal[end_peak : end_idx + 1])
    #print("start and end and START_PEAK: ", start_idx, end_idx, start_peak)
    return start_idx, end_idx



def _resolve_overlaps(seg_contractions, toco_filtered, fs):
    seg_contractions.sort(key=lambda x: x["start_idx"])
    resolved = []

    if not seg_contractions:
        return resolved

    resolved.append(seg_contractions[0])

    for k in range(1, len(seg_contractions)):
        prev = resolved[-1]
        curr = seg_contractions[k]

        if curr["start_idx"] >= prev["end_idx"]:
            # No overlap
            resolved.append(curr)
            continue

        prev_peak_idx = int(prev["peak_s"] * fs)
        curr_peak_idx = int(curr["peak_s"] * fs)

        if curr_peak_idx <= prev_peak_idx:
            continue
        else:
            search_start = prev_peak_idx
            search_end   = curr_peak_idx
            segment = toco_filtered[search_start:search_end]

            split_point = search_start + np.argmin(segment)

            prev["end_idx"]       = split_point
            prev["end_seconds"]   = split_point / fs
            prev["duration"]      = (split_point - prev["start_idx"]) / fs
            curr["start_idx"]     = split_point
            curr["start_seconds"] = split_point / fs
            curr["duration"]      = (curr["end_idx"] - split_point) / fs
            resolved.append(curr)

    return resolved


In [3]:
def split_into_20min_chunks(toco_filtered, contractions_list, fs=4.0):
    """
    Splits a long TOCO signal and its detected contractions list into 
    STRICTLY complete 20-minute windows (4800 samples each).
    Any trailing remainder shorter than 20 minutes is discarded.
    """
    chunk_duration_sec = 20 * 60  # 1200 seconds
    chunk_samples = int(chunk_duration_sec * fs)  # 4800 samples
    total_samples = len(toco_filtered)
    
    # FIX: Use floor division to only count full, complete 20-minute pieces
    num_chunks = total_samples // chunk_samples
    
    dataset_chunks = []
    
    for chunk_idx in range(num_chunks):
        start_sample = chunk_idx * chunk_samples
        end_sample = start_sample + chunk_samples # Always exactly +4800
        
        # 1. Extract the perfect 20-minute signal chunk (no padding needed anymore!)
        signal_chunk = toco_filtered[start_sample:end_sample]
        
        # 2. Gather contractions that fall completely or partially inside this window
        chunk_contractions = []
        for con in contractions_list:
            # Check if there is any intersection with the current chunk window
            if con['start_idx'] < end_sample and con['end_idx'] >= start_sample:
                
                # Symmetrically trim boundaries if a contraction hits the edge of the 20-min mark
                rel_start = max(0, int(con['start_idx'] - start_sample))
                rel_end = min(chunk_samples - 1, int(con['end_idx'] - start_sample))
                
                adjusted_contraction = {
                    'start_idx': rel_start,
                    'end_idx': rel_end,
                    'start_seconds': float(rel_start / fs),
                    'end_seconds': float(rel_end / fs),
                    'duration': float((rel_end - rel_start) / fs),
                    'peak_s': float(max(0.0, con['peak_s'] - (start_sample / fs)))
                }
                chunk_contractions.append(adjusted_contraction)

        # 3. Append compiled chunk data
        dataset_chunks.append({
            'chunk_index': chunk_idx,
            'signal_values': signal_chunk,         # Guaranteed length: 4800
            'contractions': chunk_contractions     
        })
        
    return dataset_chunks

In [4]:
def create_model_training_pairs(dataset_chunks, chunk_samples=4800):
    """
    Transforms the windowed chunks into aligned Input and Target pairs for TocoNet.
    """
    X_signals = []
    Y_masks = []
    
    for chunk in dataset_chunks:
        # List 1: The signal values
        X_signals.append(chunk['signal_values'])
        
        # List 2: Generate the 0s and 1s mask matching your idea!
        mask = np.zeros(chunk_samples, dtype=np.float32)
        for con in chunk['contractions']:
            start = con['start_idx']
            end = con['end_idx']
            mask[start:end+1] = 1.0  # Fills the contraction window with 1s
            
        Y_masks.append(mask)
        
    return np.array(X_signals), np.array(Y_masks)

In [5]:
import json
import numpy as np

def save_chunks_to_json(dataset_chunks, output_filepath):
    """
    Saves the strict 20-minute signal chunks and all calculated contraction metrics
    to a formatted JSON file for convenient hand-corrections.
    """
    serializable_chunks = []
    
    for chunk in dataset_chunks:
        # Convert the 4800 numpy array signal values to a standard Python float list
        signal_list = chunk['signal_values'].tolist()
        
        # Defensive check: clear any residual NaN values to null for strict valid JSON compatibility
        signal_list = [None if np.isnan(v) else float(v) for v in signal_list]
        
        formatted_contractions = []
        for con in chunk['contractions']:
            formatted_contractions.append({
                "start_idx": int(con['start_idx']),
                "end_idx": int(con['end_idx']),
                "start_seconds": float(con['start_seconds']),
                "end_seconds": float(con['end_seconds']),
                "duration_seconds": float(con['duration']),
                "peak_seconds": float(con['peak_s'])
            })
            
        serializable_chunks.append({
            "chunk_index": int(chunk['chunk_index']),
            "signal_values": signal_list,
            "contractions": formatted_contractions
        })
    
    # Write file out with clean indentation
    with open(output_filepath, 'w', encoding='utf-8') as f:
        json.dump(serializable_chunks, f, indent=4, ensure_ascii=False)
        
    print(f"Successfully generated dataset JSON for corrections: {output_filepath}")

In [6]:
import numpy as np
import pandas as pd

# ─── 1. YOUR EXISTING MAIN BLOCK ───

cts_file = "../Data/Num1_RData.cts"
df = read_cts_file(cts_file)

# Find the valid recording start index and extract raw signal slice
start_index   = trim_by_derivative(df["TOCO"].values)
raw_toco      = df['TOCO'].where(df["TOCOSIG"] == 2).values[start_index:]

# Filter the signal
filtered_toco = apply_butterworth_filter(raw_toco, cutoff=0.02, fs=4.0)

# Segment signal at jumps and run contraction detection
toco_segments = get_toco_segments(raw_toco, filtered_toco)
contractions  = find_contractions(filtered_toco, toco_segments, threshold=15)


# ─── 2. NEW: CHUNKING & DEEP LEARNING PREPARATION ───

# Step A: Slice the continuous signal and its contractions into 20-minute chunks
# This maps the absolute indices over to relative positions (0 to 4799)
chunks = split_into_20min_chunks(filtered_toco, contractions, fs=4.0)

# Step B: Transform those chunks into paired numpy arrays for model training
# X_train will hold the 4800 signal values, Y_train will hold the 4800 binary mask values (0s and 1s)
X_train, Y_train = create_model_training_pairs(chunks, chunk_samples=4800)


# ─── 3. VERIFY YOUR OUTPUTS ───

print(f"Total 20-minute blocks generated: {len(chunks)}")
print(f"Input Data Shape (X_train): {X_train.shape}")  # Expects: (Num_Chunks, 4800)
print(f"Target Mask Shape (Y_train): {Y_train.shape}")  # Expects: (Num_Chunks, 4800)

# Quick check on the first chunk to ensure your mask generation worked:
if len(chunks) > 0:
    print(f"\nFirst chunk contains {len(chunks[0]['contractions'])} contraction(s).")
    print(f"Sum of 1s in the first target mask: {np.sum(Y_train[0])} samples.")

Successfully generated dataset JSON for corrections: ../Data/Num1_RData_chunks_to_correct.json
Total 20-minute blocks generated: 15
Input Data Shape (X_train): (15, 4800)
Target Mask Shape (Y_train): (15, 4800)

First chunk contains 7 contraction(s).
Sum of 1s in the first target mask: 2892.0 samples.


In [37]:
contractions

[{'start_seconds': np.float64(77.25),
  'end_seconds': np.float64(180.75),
  'start_idx': np.int64(309),
  'end_idx': np.int64(723),
  'duration': np.float64(103.5),
  'peak_s': np.float64(138.25)},
 {'start_seconds': np.float64(384.25),
  'end_seconds': np.float64(466.5),
  'start_idx': np.int64(1537),
  'end_idx': np.int64(1866),
  'duration': np.float64(82.25),
  'peak_s': np.float64(422.5)},
 {'start_seconds': np.float64(517.75),
  'end_seconds': np.float64(604.25),
  'start_idx': np.int64(2071),
  'end_idx': np.int64(2417),
  'duration': np.float64(86.5),
  'peak_s': np.float64(554.75)},
 {'start_seconds': np.float64(648.75),
  'end_seconds': np.float64(722.5),
  'start_idx': np.int64(2595),
  'end_idx': np.int64(2890),
  'duration': np.float64(73.75),
  'peak_s': np.float64(682.5)},
 {'start_seconds': np.float64(775.5),
  'end_seconds': np.float64(865.0),
  'start_idx': np.int64(3102),
  'end_idx': np.int64(3460),
  'duration': np.float64(89.5),
  'peak_s': np.float64(810.5)},
 {

In [33]:
chunks

[{'chunk_index': 0,
  'signal_values': array([52.82218151, 52.75858566, 52.6923438 , ..., 13.6920643 ,
         13.65598702, 13.61865472], shape=(4800,)),
  'contractions': [{'start_idx': 309,
    'end_idx': 723,
    'start_seconds': 77.25,
    'end_seconds': 180.75,
    'duration': 103.5,
    'peak_s': 138.25},
   {'start_idx': 1537,
    'end_idx': 1866,
    'start_seconds': 384.25,
    'end_seconds': 466.5,
    'duration': 82.25,
    'peak_s': 422.5},
   {'start_idx': 2071,
    'end_idx': 2417,
    'start_seconds': 517.75,
    'end_seconds': 604.25,
    'duration': 86.5,
    'peak_s': 554.75},
   {'start_idx': 2595,
    'end_idx': 2890,
    'start_seconds': 648.75,
    'end_seconds': 722.5,
    'duration': 73.75,
    'peak_s': 682.5},
   {'start_idx': 3102,
    'end_idx': 3460,
    'start_seconds': 775.5,
    'end_seconds': 865.0,
    'duration': 89.5,
    'peak_s': 810.5},
   {'start_idx': 3546,
    'end_idx': 3976,
    'start_seconds': 886.5,
    'end_seconds': 994.0,
    'duration

In [ ]:
''' 
1 - Split the data 20-20min (list inside list)
2? - Change data visualization to see the idx and be easier to correct it
3 - 



'''

In [21]:
contractions

[{'start_seconds': np.float64(77.25),
  'end_seconds': np.float64(180.75),
  'start_idx': np.int64(309),
  'end_idx': np.int64(723),
  'duration': np.float64(103.5),
  'peak_s': np.float64(138.25)},
 {'start_seconds': np.float64(384.25),
  'end_seconds': np.float64(466.5),
  'start_idx': np.int64(1537),
  'end_idx': np.int64(1866),
  'duration': np.float64(82.25),
  'peak_s': np.float64(422.5)},
 {'start_seconds': np.float64(517.75),
  'end_seconds': np.float64(604.25),
  'start_idx': np.int64(2071),
  'end_idx': np.int64(2417),
  'duration': np.float64(86.5),
  'peak_s': np.float64(554.75)},
 {'start_seconds': np.float64(648.75),
  'end_seconds': np.float64(722.5),
  'start_idx': np.int64(2595),
  'end_idx': np.int64(2890),
  'duration': np.float64(73.75),
  'peak_s': np.float64(682.5)},
 {'start_seconds': np.float64(775.5),
  'end_seconds': np.float64(865.0),
  'start_idx': np.int64(3102),
  'end_idx': np.int64(3460),
  'duration': np.float64(89.5),
  'peak_s': np.float64(810.5)},
 {